In [1]:
import pandas as pd

In [2]:
raw_genes = pd.concat([pd.read_csv(f"all_depths/IS_raw_M5_3_{i}_gene_info.tsv",sep="\t").assign(reads_source=i) for i in [1,3,7,10,25]])
    

In [3]:
raw_genes.head()

,scaffold,gene,gene_length,coverage,breadth,breadth_minCov,nucl_diversity,start,end,direction,...,dNdS_substitutions,pNpS_variants,SNV_count,SNV_S_count,SNV_N_count,SNS_count,SNS_S_count,SNS_N_count,divergent_site_count,reads_source
0,contig_27075,contig_27075_1,351.0,8.501425,0.797721,0.672365,0.046807,598,948,-1,...,0.000000,0.393893,22.0,10.0,12.0,1.0,1.0,0.0,24.0,1
1,contig_27075,contig_27075_2,192.0,9.156250,0.630208,0.385417,0.088214,1079,1270,1,...,NaN,0.154813,12.0,7.0,5.0,0.0,0.0,0.0,15.0,1
2,contig_27075,contig_27075_3,294.0,11.040816,1.000000,1.000000,0.017268,1505,1798,1,...,NaN,0.089018,8.0,6.0,2.0,0.0,0.0,0.0,8.0,1
3,contig_27075,contig_27075_4,657.0,13.395738,1.000000,1.000000,0.031979,2052,2708,-1,...,0.309746,0.144548,44.0,30.0,14.0,2.0,1.0,1.0,47.0,1
4,contig_27075,contig_27075_5,1431.0,8.952481,0.999301,0.926625,0.020670,2787,4217,-1,...,0.076823,0.116273,51.0,37.0,14.0,5.0,4.0,1.0,57.0,1


In [4]:
stb3 = pd.read_csv("all_depths/combined_reference_M5_3.stb",sep="\t",names=["scaffold","MAG"])
stb3.head()

,scaffold,MAG
0,contig_27075,M5_3.105.filtered.fa
1,contig_19530,M5_3.105.filtered.fa
2,contig_4034,M5_3.105.filtered.fa
3,contig_51476,M5_3.105.filtered.fa
4,contig_26721,M5_3.105.filtered.fa


In [5]:
raw_genes = raw_genes.set_index("scaffold").join(stb3.set_index("scaffold")).reset_index()

In [6]:
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

In [7]:
syns_text = """0.5 F
1.5 L
0.75 I
0 M
1 V
1.5 S
1 P
1 T
1 A
0.5 Y
0.5 H
0.5 Q
0.5 N
0.5 K
0.5 D
0.5 E
0.5 C
0.5 W
1.5 R
1 G"""

In [8]:
syns_table = dict((k, float(v)) for v,k in [l.split(" ") for l in syns_text.split("\n")])

In [9]:
syns_table["*"] = 0.75
syns_table

{'F': 0.5,
 'L': 1.5,
 'I': 0.75,
 'M': 0.0,
 'V': 1.0,
 'S': 1.5,
 'P': 1.0,
 'T': 1.0,
 'A': 1.0,
 'Y': 0.5,
 'H': 0.5,
 'Q': 0.5,
 'N': 0.5,
 'K': 0.5,
 'D': 0.5,
 'E': 0.5,
 'C': 0.5,
 'W': 0.5,
 'R': 1.5,
 'G': 1.0,
 '*': 0.75}

In [10]:
non_syn = dict([(k,3-v) for k, v in syns_table.items()])

In [11]:
non_syn

{'F': 2.5,
 'L': 1.5,
 'I': 2.25,
 'M': 3.0,
 'V': 2.0,
 'S': 1.5,
 'P': 2.0,
 'T': 2.0,
 'A': 2.0,
 'Y': 2.5,
 'H': 2.5,
 'Q': 2.5,
 'N': 2.5,
 'K': 2.5,
 'D': 2.5,
 'E': 2.5,
 'C': 2.5,
 'W': 2.5,
 'R': 1.5,
 'G': 2.0,
 '*': 2.25}

In [12]:
syn_pct_by_gene = dict()
nonsyn_pct_by_gene = dict()
for record in SeqIO.parse("all_depths/combined_reference_M5_3.faa", 'fasta'):
    syn_chance = sum([syns_table[aa] for aa in record.seq])
    non_syn_chance = sum([non_syn[aa] for aa in record.seq])
    syn_pct_by_gene[record.id] = syn_chance
    nonsyn_pct_by_gene[record.id] = non_syn_chance

In [13]:
raw_genes["Ls"]= raw_genes["gene"].map(syn_pct_by_gene)
raw_genes["Ln"]= raw_genes["gene"].map(nonsyn_pct_by_gene)

In [14]:
(sum(raw_genes["Ls"]) + sum(raw_genes["Ln"]))

103369332.0

In [15]:
raw_genes["gene_length"].sum()

np.float64(103212291.0)

In [16]:
len(raw_genes["gene_length"])

143093

In [17]:
raw_genes.loc[~raw_genes[["gene_length","Ls","Ln"]].apply(lambda x: x["gene_length"] == (x["Ls"] + x["Ln"]) if x["gene_length"] else True, axis=1)]

,scaffold,gene,gene_length,coverage,breadth,breadth_minCov,nucl_diversity,start,end,direction,...,SNV_S_count,SNV_N_count,SNS_count,SNS_S_count,SNS_N_count,divergent_site_count,reads_source,MAG,Ls,Ln
20018,contig_40579,contig_40579_1,NaN,3.216418,0.695274,0.179104,0.001701,1,804,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,1,M5_3.39.filtered.fa,222.00,582.00
20019,contig_40579,contig_40579_2,NaN,1.676768,0.939394,0.000000,NaN,1009,1206,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,1,M5_3.39.filtered.fa,57.25,140.75
20020,contig_40579,contig_40579_3,NaN,1.991111,0.895556,0.000000,NaN,1504,1953,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,1,M5_3.39.filtered.fa,125.50,324.50
20021,contig_40579,contig_40579_4,NaN,0.565891,0.565891,0.000000,NaN,2488,2745,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,1,M5_3.39.filtered.fa,78.00,180.00
20022,contig_40579,contig_40579_9,NaN,0.133333,0.133333,0.000000,NaN,4415,4564,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,1,M5_3.39.filtered.fa,44.00,106.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141742,contig_42439,contig_42439_3,NaN,1.804878,0.613821,NaN,NaN,382,627,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,25,M5_3.68.filtered.fa,77.00,169.00
141743,contig_42439,contig_42439_4,NaN,0.060940,0.021292,NaN,NaN,837,2198,1,...,NaN,NaN,NaN,NaN,NaN,NaN,25,M5_3.68.filtered.fa,407.75,954.25
141744,contig_42439,contig_42439_5,NaN,0.087960,0.087960,NaN,NaN,2199,7007,1,...,NaN,NaN,NaN,NaN,NaN,NaN,25,M5_3.68.filtered.fa,1472.25,3336.75
141745,contig_42439,contig_42439_6,NaN,0.179739,0.179739,NaN,NaN,7384,7689,-1,...,NaN,NaN,NaN,NaN,NaN,NaN,25,M5_3.68.filtered.fa,88.25,217.75


In [18]:
raw_genes_f = raw_genes.loc[~raw_genes["SNV_S_count"].isna()]

In [19]:
import math

In [20]:
raw_genes_f["P_s"] = raw_genes_f[["SNV_S_count","Ls","coverage"]].apply(lambda x: x["SNV_S_count"]/(x["Ls"]*x["coverage"]),axis=1)

/var/folders/l3/gwy71j9d0yx_mg0ps6l2ypz0dbfqh3/T/ipykernel_62137/2324850662.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw_genes_f["P_s"] = raw_genes_f[["SNV_S_count","Ls","coverage"]].apply(lambda x: x["SNV_S_count"]/(x["Ls"]*x["coverage"]),axis=1)


In [21]:
raw_genes_f.head()

,scaffold,gene,gene_length,coverage,breadth,breadth_minCov,nucl_diversity,start,end,direction,...,SNV_N_count,SNS_count,SNS_S_count,SNS_N_count,divergent_site_count,reads_source,MAG,Ls,Ln,P_s
0,contig_27075,contig_27075_1,351.0,8.501425,0.797721,0.672365,0.046807,598,948,-1,...,12.0,1.0,1.0,0.0,24.0,1,M5_3.105.filtered.fa,107.75,243.25,0.010917
1,contig_27075,contig_27075_2,192.0,9.156250,0.630208,0.385417,0.088214,1079,1270,1,...,5.0,0.0,0.0,0.0,15.0,1,M5_3.105.filtered.fa,49.00,143.00,0.015602
2,contig_27075,contig_27075_3,294.0,11.040816,1.000000,1.000000,0.017268,1505,1798,1,...,2.0,0.0,0.0,0.0,8.0,1,M5_3.105.filtered.fa,77.00,217.00,0.007058
3,contig_27075,contig_27075_4,657.0,13.395738,1.000000,1.000000,0.031979,2052,2708,-1,...,14.0,2.0,1.0,1.0,47.0,1,M5_3.105.filtered.fa,189.50,467.50,0.011818
4,contig_27075,contig_27075_5,1431.0,8.952481,0.999301,0.926625,0.020670,2787,4217,-1,...,14.0,5.0,4.0,1.0,57.0,1,M5_3.105.filtered.fa,432.75,998.25,0.009550


In [22]:
raw_genes_f["P_n"] = raw_genes_f[["SNV_N_count","Ln","coverage"]].apply(lambda x: x["SNV_N_count"]/(x["Ln"]*x["coverage"]),axis=1)

/var/folders/l3/gwy71j9d0yx_mg0ps6l2ypz0dbfqh3/T/ipykernel_62137/1657369427.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw_genes_f["P_n"] = raw_genes_f[["SNV_N_count","Ln","coverage"]].apply(lambda x: x["SNV_N_count"]/(x["Ln"]*x["coverage"]),axis=1)


In [23]:
raw_genes_f.head()

,scaffold,gene,gene_length,coverage,breadth,breadth_minCov,nucl_diversity,start,end,direction,...,SNS_count,SNS_S_count,SNS_N_count,divergent_site_count,reads_source,MAG,Ls,Ln,P_s,P_n
0,contig_27075,contig_27075_1,351.0,8.501425,0.797721,0.672365,0.046807,598,948,-1,...,1.0,1.0,0.0,24.0,1,M5_3.105.filtered.fa,107.75,243.25,0.010917,0.005803
1,contig_27075,contig_27075_2,192.0,9.156250,0.630208,0.385417,0.088214,1079,1270,1,...,0.0,0.0,0.0,15.0,1,M5_3.105.filtered.fa,49.00,143.00,0.015602,0.003819
2,contig_27075,contig_27075_3,294.0,11.040816,1.000000,1.000000,0.017268,1505,1798,1,...,0.0,0.0,0.0,8.0,1,M5_3.105.filtered.fa,77.00,217.00,0.007058,0.000835
3,contig_27075,contig_27075_4,657.0,13.395738,1.000000,1.000000,0.031979,2052,2708,-1,...,2.0,1.0,1.0,47.0,1,M5_3.105.filtered.fa,189.50,467.50,0.011818,0.002236
4,contig_27075,contig_27075_5,1431.0,8.952481,0.999301,0.926625,0.020670,2787,4217,-1,...,5.0,4.0,1.0,57.0,1,M5_3.105.filtered.fa,432.75,998.25,0.009550,0.001567


In [24]:
raw_genes_f.to_csv("raw_genes_filtered_w_theta_hat.tsv",sep="\t",na_rep="NA")

In [25]:
raw_genes_25 = pd.concat([pd.read_csv(f"all_depths/IS_raw_M5_25_{i}_gene_info.tsv",sep="\t").assign(reads_source=i) for i in [1,3,5,7,10,25]])


In [26]:
syn_pct_by_gene_25 = dict()
nonsyn_pct_by_gene_25 = dict()
for record in SeqIO.parse("all_depths/combined_reference_M5_25.faa", 'fasta'):
    syn_chance = sum([syns_table[aa] for aa in record.seq])
    non_syn_chance = sum([non_syn[aa] for aa in record.seq])
    syn_pct_by_gene_25[record.id] = syn_chance
    nonsyn_pct_by_gene_25[record.id] = non_syn_chance

In [27]:
raw_genes_25["Ls"]= raw_genes_25["gene"].map(syn_pct_by_gene)
raw_genes_25["Ln"]= raw_genes_25["gene"].map(nonsyn_pct_by_gene)

In [28]:
raw_SNVs = pd.concat([pd.read_csv(f"all_depths/IS_raw_M5_3_{i}_SNVs.tsv",sep="\t").assign(reads_source=i) for i in [1,3,7,10,25]])

In [29]:
raw_SNV_site_counts = raw_SNVs.groupby(["gene","reads_source"])["mutation_type"].value_counts().unstack().reset_index()[["gene","reads_source","S","N"]].rename(columns={"S":"S_SNV_Site_count","N":"N_SNV_Site_count"})

In [30]:
raw_genes_f = pd.merge(how="left", left=raw_genes_f, right=raw_SNV_site_counts, left_on=["gene","reads_source"], right_on=["gene", "reads_source"])

In [31]:
raw_genes_f["theta_n"] = raw_genes_f[["N_SNV_Site_count","Ln","coverage"]].apply(lambda x: x["N_SNV_Site_count"]/(x["Ln"]*math.log(x["coverage"])),axis=1)

/var/folders/l3/gwy71j9d0yx_mg0ps6l2ypz0dbfqh3/T/ipykernel_62137/640367873.py:1: RuntimeWarning: divide by zero encountered in scalar divide
  raw_genes_f["theta_n"] = raw_genes_f[["N_SNV_Site_count","Ln","coverage"]].apply(lambda x: x["N_SNV_Site_count"]/(x["Ln"]*math.log(x["coverage"])),axis=1)


In [32]:
raw_genes_f["theta_s"] = raw_genes_f[["S_SNV_Site_count","Ls","coverage"]].apply(lambda x: x["S_SNV_Site_count"]/(x["Ls"]*math.log(x["coverage"])),axis=1)

/var/folders/l3/gwy71j9d0yx_mg0ps6l2ypz0dbfqh3/T/ipykernel_62137/2866317865.py:1: RuntimeWarning: divide by zero encountered in scalar divide
  raw_genes_f["theta_s"] = raw_genes_f[["S_SNV_Site_count","Ls","coverage"]].apply(lambda x: x["S_SNV_Site_count"]/(x["Ls"]*math.log(x["coverage"])),axis=1)


In [33]:
raw_genes_f["theta_ns"] = raw_genes_f["theta_n"] / raw_genes_f["theta_s"]

In [34]:
raw_genes_f.to_csv("raw_genes_filtered_w_theta_hat_M5.tsv",sep="\t",na_rep="NA")